#### FOURTH ATTEMPT

## Data Modelling

OpTc Dataset
Overview The OpTC (Operationally Transparent Cyber) dataset was developed by Five Directions, under the DARPA Transparent Computing programme, to support research into large-scale cyber-security monitoring and attack detection. It contains endpoint telemetry collected from Windows 10 computers, recording system and network activity through eCAR (extended Cyber Analytics Repository) events. This dataset contains records, including things such as processes, files, network flows, registry activity and other host events. The original release contains roughly a terabyte of compressed data from hundreds of hosts. It includes benign activity as well as red-team attack activity.

I will be using the corrected 2026 version of the OpTC dataset. INRIA reports that the original dataset contains errors involving unique identifiers and other event properties and recommends using the corrected version instead. It is available at: https://entrepot.recherche.data.gouv.fr/dataset.xhtml?persistentId=doi%3A10.57745%2FUXCWOC&utm_source=chatgpt.com

**Project goal:**

The goal of this project is to investigate the use of ML/DL to predict the occurrence of cybersecurity attacks. The project will first analyse and preprocess the OpTC event data to identify behavioural patterns associated with malicious activity, before transforming the sequential telemetry into suitable features and time-based samples for modelling. Different ML/DL approaches will then be evaluated to determine whether patterns in system and network activity can provide sufficient information to identify or predict an impending cyber attack.

**Chapter goal:**

The goal of this chapter is to use the cleaned OpTC telemetry generated in the previous chapter to train and evaluate ML/DL models for malicious event detection. The models will attempt to classify individual telemetry events as benign or malicious. This also allows me to apply the modelling pipeline used for the previous datasets to the more complex OpTC telemetry and compare model performance before moving on to the main attack prediction experiment.

In [120]:
# Imports
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import random
import tensorflow as tf

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Libraries for models
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Libraries for evaluation
from sklearn import metrics

# Libraries for deep learning
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.layers import Conv1D, MaxPooling1D, Flatten
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.optimizers import Adam

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from pathlib import Path
import pyarrow.parquet as pq


# Ignore warnings
import warnings
warnings.filterwarnings("ignore")





In [121]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Create and save a smaller OpTC sample

In [122]:

path = Path("/content/drive/MyDrive/solutions")

# Location of ready flattened parquet files in Google Drive
data_path = Path("/content/drive/MyDrive/solutions/OpTC_ready")

 # Specify the location where the sample will be stored
sample2_path = path / "OpTC_sample2_cleaned"

# Get all parquet files
files = list(data_path.glob("*.parquet"))

sample2_path.mkdir(parents=True, exist_ok=True)


# Select attack hosts and corresponding control hosts
selected_hosts = [
    # Attack hosts
    # "sysclient0051",
    # "sysclient0811",

    "sysclient0201", # September 23
    "sysclient0501", # September 24
    "sysclient0351", # September 25 (for testing)
    "sysclient0051", # September 25 (for training)

    # Control hosts
    "sysclient0352",
    "sysclient0075"
]


# Get only the parquet files belonging to the selected hosts
selected_files = [
    file for file in files
    if any(host in file.name.lower() for host in selected_hosts)
]

print(f"Selected files: {len(selected_files)}")



# Data Cleaning (Same as third_attempt)
# =====================================
missing_counts = {}
total_rows = 0

for file in files:

    parquet_file = pq.ParquetFile(file)
    metadata = parquet_file.metadata

    total_rows += metadata.num_rows

    for i in range(metadata.num_columns):

        column_name = metadata.schema.column(i).name
        null_count = 0

        for row_group in range(metadata.num_row_groups):

            stats = metadata.row_group(row_group).column(i).statistics

            if stats is not None and stats.null_count is not None:
                null_count += stats.null_count

        missing_counts[column_name] = (
            missing_counts.get(column_name, 0) + null_count
        )
# Calculate percentage of missing values
missing_percentage = (
    pd.Series(missing_counts) / total_rows * 100
).sort_values(ascending=False)


high_missing_cols = missing_percentage[missing_percentage > 99].index.tolist()

print(f"Columns with >99% missing values: {len(high_missing_cols)}")
print(high_missing_cols)


# Specify the columns to drop
# i.e. columns with many missing and identifier variables
drop_cols = set(
    high_missing_cols +
    ["id", "actorID", "objectID"]
)

# ========================================================================

# Process the selected files one at a time to reduce RAM usage
for i, file in enumerate(selected_files, start=1):

    print(f"\n[{i}/{len(selected_files)}] Processing {file.name}")

    # Load one host at a time
    data = pd.read_parquet(file)

    # Drop highly missing and identifier columns
    data = data.drop(
        columns=drop_cols,
        errors="ignore"
    )

    # Convert timestamp to datetime
    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        utc=True
    )

    # Arrange each host's events in chronological order
    data = data.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    # Save each cleaned host separately
    output_file = sample2_path / file.name

    data.to_parquet(
        output_file,
        index=False
    )

    print(
        f"Saved {file.name} | "
        f"{len(data):,} rows | "
        f"{data.shape[1]} columns"
    )

    # Clear the current file from memory before loading the next
    del data


print("\nDone.")

Selected files: 7
Columns with >99% missing values: 22
['requesting_domain', 'requesting_user', 'privileges', 'user_name', 'requesting_logon_id', 'logon_id', 'task_pid', 'task_process_uuid', 'path', 'task_name', 'payload', 'context_info', 'sid', 'user', 'tgt_pid_uuid', 'type', 'value', 'data', 'key', 'new_path', 'start_time', 'end_time']

[1/7] Processing 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet
Saved 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet | 136,241 rows | 37 columns

[2/7] Processing 2019-09-25_AIA-351-375_sysclient0352.parquet
Saved 2019-09-25_AIA-351-375_sysclient0352.parquet | 2,476,116 rows | 34 columns

[3/7] Processing 2019-09-25_AIA-351-375_sysclient0351.parquet
Saved 2019-09-25_AIA-351-375_sysclient0351.parquet | 2,518,452 rows | 34 columns

[4/7] Processing 2019-09-24_AIA-501-525_sysclient0501.parquet
Saved 2019-09-24_AIA-501-525_sysclient0501.parquet | 4,383,849 rows | 34 columns

[5/7] Processing 2019-09-23_AIA-201-2

### 2. Load and preview the data

In [123]:
# Location of ready flattened parquet files in Google Drive
data_path = Path("/content/drive/MyDrive/solutions/OpTC_sample2_cleaned")

# Get the parquet files
files = list(data_path.glob("*.parquet"))
print(f"Number of files: {len(files)}")



# Check the number of rows in each file
for file in files:

    data = pd.read_parquet(
        file,
        columns=["hostname", "label"]
    )

    print(
        file.name,
        "| Rows:", f"{len(data):,}",
        "| Malicious:", f"{data['label'].sum():,}"
    )


Number of files: 7
2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet | Rows: 136,241 | Malicious: 0
2019-09-25_AIA-351-375_sysclient0352.parquet | Rows: 2,476,116 | Malicious: 0
2019-09-25_AIA-351-375_sysclient0351.parquet | Rows: 2,518,452 | Malicious: 18,889
2019-09-24_AIA-501-525_sysclient0501.parquet | Rows: 4,383,849 | Malicious: 27,577
2019-09-23_AIA-201-225_sysclient0201.parquet | Rows: 4,376,026 | Malicious: 26,681
2019-09-25_AIA-51-75_sysclient0051.parquet | Rows: 2,524,447 | Malicious: 5,145
2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0075.parquet | Rows: 140,723 | Malicious: 0


### 2. Further Data Inspection

In [124]:
# Load one cleaned file to inspect the available features
data = pd.read_parquet(files[0])

print("Columns:")
print(data.columns.tolist())

print("\nData types:")
print(data.dtypes)

Columns:
['action', 'hostname', 'object', 'pid', 'ppid', 'principal', 'tid', 'timestamp', 'label', 'acuity_level', 'base_address', 'command_line', 'dest_ip', 'dest_port', 'direction', 'file_path', 'image_path', 'info_class', 'l4protocol', 'module_path', 'name', 'parent_image_path', 'service_type', 'size', 'src_ip', 'src_pid', 'src_port', 'src_tid', 'stack_base', 'stack_limit', 'start_address', 'start_type', 'subprocess_tag', 'tgt_pid', 'tgt_tid', 'user_stack_base', 'user_stack_limit']

Data types:
action                            object
hostname                          object
object                            object
pid                                int64
ppid                               int64
principal                         object
tid                                int64
timestamp            datetime64[ns, UTC]
label                              int64
acuity_level                      object
base_address                      object
command_line                      object
dest_

In [125]:
# Identify numerical columns
numeric_cols = data.select_dtypes(
    include=np.number
).columns

print("Numerical columns:")
print(numeric_cols.tolist())

Numerical columns:
['pid', 'ppid', 'tid', 'label']


In [126]:
# Identify categorical columns
categorical_cols = data.select_dtypes(
    include=["object", "string"]
).columns

# Get the number of unique values of categorical variables
# so as to know if encoding would make data explode

print("Categorical columns and unique values:")

for col in categorical_cols:
    print(f"{col}: {data[col].nunique():,}")

Categorical columns and unique values:
action: 20
hostname: 1
object: 10
principal: 8
acuity_level: 6
base_address: 1,116
command_line: 74
dest_ip: 21
dest_port: 102
direction: 2
file_path: 956
image_path: 56
info_class: 7
l4protocol: 4
module_path: 1,344
name: 1
parent_image_path: 41
service_type: 1
size: 1,338
src_ip: 5,039
src_pid: 82
src_port: 5,125
src_tid: 930
stack_base: 647
stack_limit: 647
start_address: 145
start_type: 1
subprocess_tag: 43
tgt_pid: 91
tgt_tid: 893
user_stack_base: 1,388
user_stack_limit: 1,421


### 2. Initital feature Selection

In [127]:
# Select features for the initial detection experiment
# I am doing this to save RAM actually
# I had to include hostname, but will remove it later after the split

selected_features = [
    "action",
    "object",
    "acuity_level",
    "direction",
    "info_class",
    "l4protocol",
    "pid",
    "ppid",
    "tid",
    "timestamp",
    "label",
    "hostname"
]



### 3. Creating a DataFrame of OpTC

In [128]:
# Load only the selected features from each file
# and use this to create a dataframe


data_parts = []

for file in files:

    part = pd.read_parquet(
        file,
        columns=selected_features
    )

    data_parts.append(part)


# Combine the selected data
data = pd.concat(
    data_parts,
    ignore_index=True
)

del data_parts

print(f"Dataset shape: {data.shape}")

print("\nLabel distribution:")
print(data["label"].value_counts())

print("\nLabel percentages:")
print(data["label"].value_counts(normalize=True) * 100)

Dataset shape: (16555854, 12)

Label distribution:
label
0    16477562
1       78292
Name: count, dtype: int64

Label percentages:
label
0    99.527104
1     0.472896
Name: proportion, dtype: float64


### 4. Data Splitting (Host and time conscious)

In [129]:
# Specify the hosts to keep for testing

test_hosts = [
    "sysclient0351",
    "sysclient0352"
]

# Separate the files into training and testing files
train_files = [
    file for file in files
    if not any(host in file.name.lower() for host in test_hosts)
]

test_files = [
    file for file in files
    if any(host in file.name.lower() for host in test_hosts)
]

print(f"Training files: {len(train_files)}")
print(f"Testing files: {len(test_files)}")

Training files: 5
Testing files: 2


### 5. Separate maliciouss and benig parts

In [130]:
# Features needed for the detection experiment
model_features = [
    "action",
    "object",
    "acuity_level",
    "direction",
    "info_class",
    "l4protocol",
    "pid",
    "ppid",
    "tid",
    "timestamp",
    "label"
]


# Store malicious and benign events separately
malicious_parts = []
benign_parts = []

for file in train_files:

    print(f"Processing: {file.name}")

    part = pd.read_parquet(
        file,
        columns=model_features
    )

    # Keep all malicious events
    malicious_part = part[
        part["label"] == 1
    ]

    if len(malicious_part) > 0:
        malicious_parts.append(malicious_part)

    # Store benign events temporarily
    benign_part = part[
        part["label"] == 0
    ]

    benign_parts.append(benign_part)

    del part

Processing: 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet
Processing: 2019-09-24_AIA-501-525_sysclient0501.parquet
Processing: 2019-09-23_AIA-201-225_sysclient0201.parquet
Processing: 2019-09-25_AIA-51-75_sysclient0051.parquet
Processing: 2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0075.parquet


In [131]:
print(f"Dataset shape: {data.shape}")

print("\nLabel distribution:")
print(data["label"].value_counts())

print("\nLabel percentages:")
print(data["label"].value_counts(normalize=True) * 100)

benign = (data["label"] == 0).sum()
malicious = (data["label"] == 1).sum()

print(f"\nBenign-to-malicious ratio: {benign / malicious:.2f}:1")

Dataset shape: (16555854, 12)

Label distribution:
label
0    16477562
1       78292
Name: count, dtype: int64

Label percentages:
label
0    99.527104
1     0.472896
Name: proportion, dtype: float64

Benign-to-malicious ratio: 210.46:1


### 6. Create Split Samples

In [132]:
total_malicious = sum(
    pd.read_parquet(file, columns=["label"])["label"].sum()
    for file in train_files
)

print("TOTAL MALICIOUS EXPECTED:", total_malicious)

train_parts = []

for file in train_files:

    part = pd.read_parquet(
        file,
        columns=selected_features
    )

    malicious_part = part[part["label"] == 1]
    benign_part = part[part["label"] == 0]

    print(
        file.name,
        "| malicious:", len(malicious_part),
        "| benign:", len(benign_part)
    )

    benign_sample = benign_part.sample(
        n=min(len(benign_part), total_malicious),
        random_state=7
    )

    train_parts.append(
        pd.concat(
            [malicious_part, benign_sample],
            ignore_index=True
        )
    )

train_data = pd.concat(
    train_parts,
    ignore_index=True
)

print("\nAFTER CONCAT:")
print(train_data["label"].value_counts())

TOTAL MALICIOUS EXPECTED: 59403
2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0051.parquet | malicious: 0 | benign: 136241
2019-09-24_AIA-501-525_sysclient0501.parquet | malicious: 27577 | benign: 4356272
2019-09-23_AIA-201-225_sysclient0201.parquet | malicious: 26681 | benign: 4349345
2019-09-25_AIA-51-75_sysclient0051.parquet | malicious: 5145 | benign: 2519302
2019-09-16_AIA-51-75_AIA-51-75.ecar-2019-09-16-sysclient0075.parquet | malicious: 0 | benign: 140723

AFTER CONCAT:
label
0    297015
1     59403
Name: count, dtype: int64


In [133]:
# Load the held-out test hosts
test_data = pd.concat(
    [pd.read_parquet(file, columns=selected_features) for file in test_files],
    ignore_index=True
)

print(test_data["label"].value_counts())

# Extract time features
for dataset in [train_data, test_data]:
    dataset["hour"] = dataset["timestamp"].dt.hour
    dataset["minute"] = dataset["timestamp"].dt.minute

# Define independent and dependent variables
x_train = train_data.drop(columns=["label", "timestamp"])
y_train = train_data["label"]

x_test = test_data.drop(columns=["label", "timestamp"])
y_test = test_data["label"]

label
0    4975679
1      18889
Name: count, dtype: int64


### 7. Prepare train and test data

In [134]:
# Extract time features
for dataset in [train_data, test_data]:
    dataset["hour"] = dataset["timestamp"].dt.hour
    dataset["minute"] = dataset["timestamp"].dt.minute

# Remove timestamp and hostname, then separate features and labels
x_train = train_data.drop(columns=["label", "timestamp", "hostname"], errors="ignore")
y_train = train_data["label"]

x_test = test_data.drop(columns=["label", "timestamp", "hostname"], errors="ignore")
y_test = test_data["label"]

### 8. Encode Categorical Features

In [135]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify categorical and numerical columns
cat_cols = x_train.select_dtypes(include=["object"]).columns
numeric_cols = x_train.select_dtypes(include=np.number).columns

# Encode categorical features and scale numerical features
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), cat_cols),
    ("num", StandardScaler(), numeric_cols)
], sparse_threshold=1.0)

X = preprocessor.fit_transform(x_train).astype(np.float32)
X_TEST = preprocessor.transform(x_test).astype(np.float32)

Y = y_train.copy()
Y_TEST = y_test.copy()

print(X.shape)
print(X_TEST.shape)

(356418, 60)
(4994568, 60)


### 9. Define and Train ML models

In [136]:
# Define models
models = [
    ("Logistic Regression", LogisticRegression(
        max_iter=1000,
        random_state=7,
        class_weight="balanced"
    )),

    ("Bernoulli NB", BernoulliNB()),

    ("Decision Tree", DecisionTreeClassifier(
        random_state=7,
        class_weight="balanced",
        max_depth=10
    )),

    ("Random Forest", RandomForestClassifier(
        n_estimators=100,
        random_state=7,
        class_weight="balanced",
        n_jobs=-1
    )),

    ("XGBoost", XGBClassifier(
        random_state=7,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss",
        n_jobs=-1
    ))
]

In [137]:
# Train and evaluate the models on training data

for name, model in models:

    # Train model
    model.fit(X, Y)

    # Make predictions on training data
    predictions = model.predict(X)

    accuracy = metrics.accuracy_score(Y, predictions)
    conf_matrix = metrics.confusion_matrix(Y, predictions)
    report = metrics.classification_report(Y, predictions)

    print(f"\n===== {name} - Train Evaluation =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Logistic Regression - Train Evaluation =====
Accuracy: 0.9399637504278684
Confusion Matrix:
 [[277947  19068]
 [  2330  57073]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.94      0.96    297015
           1       0.75      0.96      0.84     59403

    accuracy                           0.94    356418
   macro avg       0.87      0.95      0.90    356418
weighted avg       0.95      0.94      0.94    356418


===== Bernoulli NB - Train Evaluation =====
Accuracy: 0.8843408582058145
Confusion Matrix:
 [[265104  31911]
 [  9312  50091]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.89      0.93    297015
           1       0.61      0.84      0.71     59403

    accuracy                           0.88    356418
   macro avg       0.79      0.87      0.82    356418
weighted avg       0.91      0.88      0.89    356418


===== Decision Tree - Train Evalua

### 10. Validate on Test data

In [138]:
# Evaluate the models on test data

for name, model in models:

    predictions = model.predict(X_TEST)

    accuracy = metrics.accuracy_score(Y_TEST, predictions)
    conf_matrix = metrics.confusion_matrix(Y_TEST, predictions)
    report = metrics.classification_report(Y_TEST, predictions)

    print(f"\n===== {name} - Test Evaluation =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Logistic Regression - Test Evaluation =====
Accuracy: 0.942260471776538
Confusion Matrix:
 [[4702491  273188]
 [  15196    3693]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.97   4975679
           1       0.01      0.20      0.02     18889

    accuracy                           0.94   4994568
   macro avg       0.51      0.57      0.50   4994568
weighted avg       0.99      0.94      0.97   4994568


===== Bernoulli NB - Test Evaluation =====
Accuracy: 0.939049583467479
Confusion Matrix:
 [[4679947  295732]
 [   8689   10200]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97   4975679
           1       0.03      0.54      0.06     18889

    accuracy                           0.94   4994568
   macro avg       0.52      0.74      0.52   4994568
weighted avg       0.99      0.94      0.97   4994568


===== Decision Tree - Test Eva

### 11. DL Models - ANN

In [139]:
# Reduce the data for ANN/CNN because the full dataset exceeds RAM
np.random.seed(7)

train_idx = np.random.choice(X.shape[0], 200000, replace=False)
test_idx = np.random.choice(X_TEST.shape[0], 200000, replace=False)

X = X[train_idx].toarray().astype("float32")
Y = Y.iloc[train_idx]

X_TEST = X_TEST[test_idx].toarray().astype("float32")
Y_TEST = Y_TEST.iloc[test_idx]



In [140]:
# Build ANN
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train ANN
history = model.fit(
    X, Y,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

# Train evaluation
train_pred = (model.predict(X) > 0.5).astype(int)

print("Train Accuracy:", metrics.accuracy_score(Y, train_pred))
print(metrics.classification_report(Y, train_pred))

# Test evaluation
test_pred = (model.predict(X_TEST) > 0.5).astype(int)

print("Test Accuracy:", metrics.accuracy_score(Y_TEST, test_pred))
print(metrics.classification_report(Y_TEST, test_pred))

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9765 - loss: 0.0592 - val_accuracy: 0.9865 - val_loss: 0.0311
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9848 - loss: 0.0354 - val_accuracy: 0.9901 - val_loss: 0.0240
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9879 - loss: 0.0292 - val_accuracy: 0.9906 - val_loss: 0.0218
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9889 - loss: 0.0260 - val_accuracy: 0.9912 - val_loss: 0.0199
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9904 - loss: 0.0240 - val_accuracy: 0.9936 - val_loss: 0.0171
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9911 - loss: 0.0218 - val_accuracy: 0.9941 - val_loss: 0.0158
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9919 - loss: 0.0201 - val_accuracy: 0.9948 - val_loss: 0.0142
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9921 - loss: 0.0197 - 

### 12. DL Models - CNN

In [141]:
# Reshape input for Conv1D
X_cnn = np.expand_dims(X, axis=2)
X_TEST_cnn = np.expand_dims(X_TEST, axis=2)


# Build the CNN
cnn_classifier = Sequential()

cnn_classifier.add(
    Conv1D(
        32,
        kernel_size=3,
        activation='relu',
        padding='same',
        input_shape=(X_cnn.shape[1], 1)
    )
)

cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(
    Conv1D(
        64,
        kernel_size=3,
        activation='relu',
        padding='same'
    )
)

cnn_classifier.add(BatchNormalization())
cnn_classifier.add(MaxPooling1D(pool_size=2))
cnn_classifier.add(Dropout(0.2))

cnn_classifier.add(Flatten())
cnn_classifier.add(Dense(64, activation='relu'))
cnn_classifier.add(Dropout(0.3))

cnn_classifier.add(Dense(1, activation='sigmoid'))


# Compile the model
cnn_classifier.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# Train the model
cnn_history = cnn_classifier.fit(
    X_cnn,
    Y,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)


# Train evaluation
train_pred = (cnn_classifier.predict(X_cnn) > 0.5).astype(int)

print("Train Accuracy:", metrics.accuracy_score(Y, train_pred))
print(metrics.classification_report(Y, train_pred))


# Test evaluation
test_pred = (cnn_classifier.predict(X_TEST_cnn) > 0.5).astype(int)

print("Test Accuracy:", metrics.accuracy_score(Y_TEST, test_pred))
print(metrics.classification_report(Y_TEST, test_pred))

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 28s 10ms/step - accuracy: 0.9745 - loss: 0.0628 - val_accuracy: 0.9859 - val_loss: 0.0329
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 25s 10ms/step - accuracy: 0.9854 - loss: 0.0352 - val_accuracy: 0.9894 - val_loss: 0.0258
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 26s 10ms/step - accuracy: 0.9882 - loss: 0.0291 - val_accuracy: 0.9919 - val_loss: 0.0181
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 26s 10ms/step - accuracy: 0.9898 - loss: 0.0253 - val_accuracy: 0.9931 - val_loss: 0.0163
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 26s 10ms/step - accuracy: 0.9913 - loss: 0.0225 - val_accuracy: 0.9949 - val_loss: 0.0141
Epoch 6/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 40s 10ms/step - accuracy: 0.9923 - loss: 0.0207 - val_accuracy: 0.9954 - val_loss: 0.0123
Epoch 7/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 25s 10ms/step - accuracy: 0.9927 - loss: 0.0191 - val_accuracy: 0.9947 - val_loss: 0.0138
Epoch 8/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 25s 10ms/step - accuracy: 0.9930 -

### 13. Results Presentation

In [142]:
# Generate summary tables for presentation

train_results = []
test_results = []

# Save ANN model before looping through classical models
ann_classifier = model


# Classical ML models
for name, classifier in models:

    train_predictions = classifier.predict(X)
    test_predictions = classifier.predict(X_TEST)

    train_results.append([
        name,
        metrics.accuracy_score(Y, train_predictions),
        metrics.precision_score(Y, train_predictions, average="macro", zero_division=0),
        metrics.recall_score(Y, train_predictions, average="macro", zero_division=0),
        metrics.f1_score(Y, train_predictions, average="macro", zero_division=0)
    ])

    test_results.append([
        name,
        metrics.accuracy_score(Y_TEST, test_predictions),
        metrics.precision_score(Y_TEST, test_predictions, average="macro", zero_division=0),
        metrics.recall_score(Y_TEST, test_predictions, average="macro", zero_division=0),
        metrics.f1_score(Y_TEST, test_predictions, average="macro", zero_division=0)
    ])


# ANN predictions
ann_train_pred = (
    ann_classifier.predict(X, verbose=0) > 0.5
).astype(int).ravel()

ann_test_pred = (
    ann_classifier.predict(X_TEST, verbose=0) > 0.5
).astype(int).ravel()

train_results.append([
    "ANN",
    metrics.accuracy_score(Y, ann_train_pred),
    metrics.precision_score(Y, ann_train_pred, average="macro", zero_division=0),
    metrics.recall_score(Y, ann_train_pred, average="macro", zero_division=0),
    metrics.f1_score(Y, ann_train_pred, average="macro", zero_division=0)
])

test_results.append([
    "ANN",
    metrics.accuracy_score(Y_TEST, ann_test_pred),
    metrics.precision_score(Y_TEST, ann_test_pred, average="macro", zero_division=0),
    metrics.recall_score(Y_TEST, ann_test_pred, average="macro", zero_division=0),
    metrics.f1_score(Y_TEST, ann_test_pred, average="macro", zero_division=0)
])


# CNN predictions
cnn_train_pred = (
    cnn_classifier.predict(X_cnn, verbose=0) > 0.5
).astype(int).ravel()

cnn_test_pred = (
    cnn_classifier.predict(X_TEST_cnn, verbose=0) > 0.5
).astype(int).ravel()

train_results.append([
    "CNN",
    metrics.accuracy_score(Y, cnn_train_pred),
    metrics.precision_score(Y, cnn_train_pred, average="macro", zero_division=0),
    metrics.recall_score(Y, cnn_train_pred, average="macro", zero_division=0),
    metrics.f1_score(Y, cnn_train_pred, average="macro", zero_division=0)
])

test_results.append([
    "CNN",
    metrics.accuracy_score(Y_TEST, cnn_test_pred),
    metrics.precision_score(Y_TEST, cnn_test_pred, average="macro", zero_division=0),
    metrics.recall_score(Y_TEST, cnn_test_pred, average="macro", zero_division=0),
    metrics.f1_score(Y_TEST, cnn_test_pred, average="macro", zero_division=0)
])


# Create presentation tables
columns = [
    "Model",
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1"
]

train_results = pd.DataFrame(
    train_results,
    columns=columns
).round(4)

test_results = pd.DataFrame(
    test_results,
    columns=columns
).round(4)


print("TRAIN RESULTS")
display(train_results)

print("TEST RESULTS")
display(test_results)

TRAIN RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.9395,0.8693,0.9480,0.9016
1,Bernoulli NB,0.8850,0.7888,0.8687,0.8187
2,Decision Tree,0.9953,0.9862,0.9971,0.9916
3,Random Forest,1.0000,1.0000,1.0000,1.0000
4,XGBoost,0.9868,0.9656,0.9887,0.9767
5,ANN,0.9956,0.9893,0.9948,0.9921
6,CNN,0.9968,0.9923,0.9961,0.9942


TEST RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,0.9423,0.5050,0.5753,0.4974
1,Bernoulli NB,0.9385,0.5149,0.7465,0.5139
2,Decision Tree,0.9920,0.5200,0.5281,0.5233
3,Random Forest,0.9963,0.5377,0.5020,0.5031
4,XGBoost,0.9843,0.5055,0.5193,0.5072
5,ANN,0.9926,0.5167,0.5200,0.5182
6,CNN,0.9925,0.5135,0.5164,0.5148
